# Bronze Layer - Data Ingestion

Load raw CSV files from the volume into bronze Delta tables.

In [0]:
# Read Orders.csv
orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/ecommerce_catalog/bronze/raw_data/Orders.csv")
)

# Save as Bronze table
orders_df.write.mode("overwrite").saveAsTable(
    "ecommerce_catalog.bronze.orders"
)

display(spark.table("ecommerce_catalog.bronze.orders"))

In [0]:
order_items_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/ecommerce_catalog/bronze/raw_data/Order_items.csv")
)

# Rename column to match existing table schema
order_items_df = order_items_df.withColumnRenamed("Total amount", "Total_amount")

order_items_df.write.mode("overwrite").saveAsTable(
    "ecommerce_catalog.bronze.order_items"
)

display(spark.table("ecommerce_catalog.bronze.order_items"))


In [0]:
payments_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/ecommerce_catalog/bronze/raw_data/Payments.csv")
)

payments_df = payments_df.withColumnRenamed("Amount_paid ", "Amount_paid")

payments_df.write.mode("overwrite").saveAsTable(
    "ecommerce_catalog.bronze.payments"
)

display(spark.table("ecommerce_catalog.bronze.payments"))

# Silver Layer - Data Transformations

Create cleaned, joined, and enriched tables in the silver schema with:
* Data quality validations
* Standardized column names
* Joined datasets
* Calculated fields
* Business logic

In [0]:
# Load Customers
customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/ecommerce_catalog/bronze/raw_data/Customers.csv")
)

customers_df.write.mode("overwrite").saveAsTable(
    "ecommerce_catalog.bronze.customers"
)

# Load Products
products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/ecommerce_catalog/bronze/raw_data/Products.csv")
)

products_df.write.mode("overwrite").saveAsTable(
    "ecommerce_catalog.bronze.products"
)

print("Bronze tables loaded: customers, products")

In [0]:
from pyspark.sql import functions as F

# Load bronze tables
orders = spark.table("ecommerce_catalog.bronze.orders")
customers = spark.table("ecommerce_catalog.bronze.customers")
order_items = spark.table("ecommerce_catalog.bronze.order_items")
products = spark.table("ecommerce_catalog.bronze.products")
payments = spark.table("ecommerce_catalog.bronze.payments")

# Join orders with customers
orders_enriched = orders.join(
    customers,
    "customer_id",
    "left"
).select(
    orders["*"],
    customers.name.alias("customer_name"),
    customers.email,
    customers.phone,
    customers.country
)

# Aggregate order items to get order-level metrics
order_items_agg = order_items.groupBy("order_id").agg(
    F.count("order_item_id").alias("total_items"),
    F.sum("quantity").alias("total_quantity"),
    F.sum("Total_amount").alias("items_total_amount")
)

# Join with order items aggregation
orders_enriched = orders_enriched.join(
    order_items_agg,
    "order_id",
    "left"
)

# Join with payments to get payment info
orders_enriched = orders_enriched.join(
    payments.select("order_id", "payment_method", "Amount_paid"),
    "order_id",
    "left"
)

# Add calculated fields and data quality flags
orders_enriched = orders_enriched.withColumn(
    "is_paid",
    F.when(F.col("Amount_paid").isNotNull() & (F.col("Amount_paid") > 0), True).otherwise(False)
).withColumn(
    "payment_complete",
    F.when(
        (F.col("Amount_paid").isNotNull()) & (F.col("items_total_amount").isNotNull()) & 
        (F.col("Amount_paid") >= F.col("items_total_amount")), 
        True
    ).otherwise(False)
).withColumn(
    "order_year",
    F.year("order_date")
).withColumn(
    "order_month",
    F.month("order_date")
)

# Write to silver table
orders_enriched.write.mode("overwrite").saveAsTable(
    "ecommerce_catalog.silver.orders_enriched"
)

print(f"Created silver.orders_enriched with {orders_enriched.count()} rows")
display(orders_enriched.limit(10))

In [0]:
# Create detailed order items table with product information
order_items_detailed = order_items.join(
    products,
    "product_id",
    "left"
).join(
    orders.select("order_id", "customer_id", "order_date"),
    "order_id",
    "left"
)

# Add calculated fields
order_items_detailed = order_items_detailed.withColumn(
    "line_total",
    F.col("quantity") * F.col("price_each")
).withColumn(
    "is_valid_price",
    F.when(F.col("price_each") > 0, True).otherwise(False)
).withColumn(
    "discount_amount",
    (F.col("price") - F.col("price_each")) * F.col("quantity")
).withColumn(
    "order_year",
    F.year(F.col("order_date"))
).withColumn(
    "order_month",
    F.month(F.col("order_date"))
)

# Select and rename columns for consistency
order_items_detailed = order_items_detailed.select(
    "order_item_id",
    "order_id",
    "product_id",
    "customer_id",
    "order_date",
    "order_year",
    "order_month",
    F.col("product_name"),
    F.col("category"),
    F.col("price").alias("product_list_price"),
    "quantity",
    "price_each",
    "line_total",
    "discount_amount",
    "is_valid_price"
)

# Write to silver table
order_items_detailed.write.mode("overwrite").saveAsTable(
    "ecommerce_catalog.silver.order_items_detailed"
)

print(f"Created silver.order_items_detailed with {order_items_detailed.count()} rows")
display(order_items_detailed.limit(10))

In [0]:
# Create customer summary with aggregated metrics
customer_summary = orders.join(
    customers,
    "customer_id",
    "inner"
).join(
    payments.select("order_id", "Amount_paid"),
    "order_id",
    "left"
).groupBy(
    "customer_id",
    customers.name.alias("customer_name"),
    "email",
    "phone",
    "country"
).agg(
    F.count("order_id").alias("total_orders"),
    F.sum("Amount_paid").alias("total_spent"),
    F.avg("Amount_paid").alias("avg_order_value"),
    F.min("order_date").alias("first_order_date"),
    F.max("order_date").alias("last_order_date")
)

# Add customer segmentation
customer_summary = customer_summary.withColumn(
    "customer_segment",
    F.when(F.col("total_orders") >= 5, "Loyal")
    .when(F.col("total_orders") >= 2, "Regular")
    .otherwise("New")
).withColumn(
    "days_since_last_order",
    F.datediff(F.current_date(), F.col("last_order_date"))
).withColumn(
    "customer_lifetime_days",
    F.datediff(F.col("last_order_date"), F.col("first_order_date"))
).withColumn(
    "is_active",
    F.when(F.col("days_since_last_order") <= 90, True).otherwise(False)
)

# Write to silver table
customer_summary.write.mode("overwrite").saveAsTable(
    "ecommerce_catalog.silver.customer_summary"
)

print(f"Created silver.customer_summary with {customer_summary.count()} rows")
display(customer_summary.limit(10))

In [0]:
# Summary of silver tables created
print("=" * 60)
print("Silver Layer Tables Created:")
print("=" * 60)

silver_tables = [
    "orders_enriched",
    "order_items_detailed",
    "customer_summary"
]

for table in silver_tables:
    full_name = f"ecommerce_catalog.silver.{table}"
    count = spark.table(full_name).count()
    print(f"✓ {full_name}: {count:,} rows")

print("=" * 60)

In [0]:
# Gold Layer
